In [3]:
# PFE Renault Tanger — Système d'alertes
## Notebook 05 : Alertes email automatiques + SHAP
### Objectif : envoyer un rapport quotidien intelligent par email

In [4]:
import pandas as pd
import numpy as np
import smtplib
import joblib
import shap
import matplotlib.pyplot as plt
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.image import MIMEImage
from datetime import datetime, date
import warnings
warnings.filterwarnings('ignore')

print("Librairies chargées ✓")

Librairies chargées ✓


In [5]:
# ── À MODIFIER AVEC TES INFORMATIONS ──────────────────────
CONFIG_EMAIL = {
    'expediteur'     : 'ton.email@gmail.com',
    'mot_de_passe'   : 'ton_app_password',   # voir instructions ci-dessous
    'destinataires'  : [
        'encadrant@renault.com',
        'ton.email@gmail.com'
    ],
    'smtp_serveur'   : 'smtp.gmail.com',
    'smtp_port'      : 587
}

SEUIL_OBJECTIF = 1.25   # m³/véhicule
SEUIL_ALERTE   = 1.25 * 1.15   # +15% = anomalie

print("Configuration chargée ✓")
print()
print("⚠️  IMPORTANT — Pour utiliser Gmail :")
print("   1. Va sur myaccount.google.com")
print("   2. Sécurité → Validation en 2 étapes → Active")
print("   3. Sécurité → Mots de passe des applications")
print("   4. Crée un mot de passe pour 'Mail'")
print("   5. Colle ce mot de passe dans CONFIG_EMAIL['mot_de_passe']")

Configuration chargée ✓

⚠️  IMPORTANT — Pour utiliser Gmail :
   1. Va sur myaccount.google.com
   2. Sécurité → Validation en 2 étapes → Active
   3. Sécurité → Mots de passe des applications
   4. Crée un mot de passe pour 'Mail'
   5. Colle ce mot de passe dans CONFIG_EMAIL['mot_de_passe']


In [7]:
df = pd.read_csv("../outputs/dataset_eau_propre.csv", parse_dates=['Date'])
df_prod = df[(df['TCM'] >= 100) & (df['is_weekend'] == 0)].copy().reset_index(drop=True)

model_xgb = joblib.load("../models/model_xgboost.pkl")

# ── CORRECTION Prophet : réentraîner avec données filtrées ──
from prophet import Prophet

df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
df_prophet.columns = ['ds','y','TCM']
df_prophet = df_prophet.dropna(subset=['y'])

# Filtrer les outliers extrêmes (KPI > 3 = jours aberrants janvier)
df_prophet = df_prophet[df_prophet['y'] < 3.0].copy()

# Réentraîner Prophet sur données propres
model_prophet = Prophet(
    yearly_seasonality=False,   # pas assez de données pour annuelle
    weekly_seasonality=True,
    daily_seasonality=False,
    seasonality_mode='additive', # additif = pas de valeurs négatives
    changepoint_prior_scale=0.01 # très stable, peu de flexibilité
)
model_prophet.add_regressor('TCM')
model_prophet.fit(df_prophet)

# Recalculer SHAP
FEATURES = [
    'TCM', 'ED_Total', 'EOR_Total', 'EI_Looker', 'EP_Looker',
    'jour_semaine', 'mois', 'trimestre', 'is_lundi',
    'ratio_EOR_ED', 'taux_EP', 'taux_EI'
]
df_xgb  = df_prod[FEATURES + ['KPI_m3_veh', 'Date', 'E. Appro Looker']].dropna().copy()
X       = df_xgb[FEATURES]

explainer   = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X)

# Sauvegarder le nouveau Prophet
joblib.dump(model_prophet, "../models/model_prophet.pkl")

print("Données et modèles chargés ✓")
print(f"Données Prophet filtrées : {len(df_prophet)} jours (KPI < 3.0)")
print(f"Dernier jour disponible  : {df_xgb['Date'].max().date()}")

15:49:56 - cmdstanpy - INFO - Chain [1] start processing
15:49:56 - cmdstanpy - INFO - Chain [1] done processing


Données et modèles chargés ✓
Données Prophet filtrées : 66 jours (KPI < 3.0)
Dernier jour disponible  : 2026-04-22


In [8]:
def analyser_jour_courant(df_xgb, shap_values, model_xgb):
    """
    Analyse le dernier jour disponible dans le dataset.
    Retourne un dictionnaire avec toutes les infos du rapport.
    """
    idx      = len(df_xgb) - 1
    row      = df_xgb.iloc[idx]
    date_j   = row['Date'].date()
    kpi_reel = row['KPI_m3_veh']
    kpi_pred = model_xgb.predict(X.iloc[[idx]])[0]

    # Statut
    if kpi_reel > SEUIL_ALERTE:
        statut     = "ANOMALIE"
        statut_emoji = "🔴"
        couleur    = "#FCEBEB"
    elif kpi_reel > SEUIL_OBJECTIF:
        statut     = "ATTENTION"
        statut_emoji = "🟡"
        couleur    = "#FAEEDA"
    else:
        statut     = "NORMAL"
        statut_emoji = "🟢"
        couleur    = "#EAF3DE"

    # Top 3 causes SHAP
    contribs = pd.Series(shap_values[idx], index=FEATURES)
    noms_lisibles = {
        'TCM'          : 'Production (TCM)',
        'ED_Total'     : 'Eau déminéralisée',
        'EOR_Total'    : 'Eau osmosée recyclée',
        'EI_Looker'    : 'Eau industrielle',
        'EP_Looker'    : 'Eau potable',
        'jour_semaine' : 'Jour de semaine',
        'mois'         : 'Mois',
        'trimestre'    : 'Trimestre',
        'is_lundi'     : 'Effet lundi',
        'ratio_EOR_ED' : 'Taux recyclage EOR/ED',
        'taux_EP'      : 'Part eau potable',
        'taux_EI'      : 'Part eau industrielle'
    }

    top3 = contribs.abs().nlargest(3)
    causes = []
    for feat, _ in top3.items():
        val_shap  = contribs[feat]
        val_reelle = X.iloc[idx][feat]
        nom        = noms_lisibles.get(feat, feat)
        direction  = "↑ augmente" if val_shap > 0 else "↓ réduit"
        causes.append({
            'feature'   : nom,
            'valeur'    : val_reelle,
            'shap'      : val_shap,
            'direction' : direction
        })

    return {
        'date'       : date_j,
        'kpi_reel'   : kpi_reel,
        'kpi_pred'   : kpi_pred,
        'tcm'        : row['TCM'],
        'conso'      : row['E. Appro Looker'],
        'statut'     : statut,
        'statut_emoji': statut_emoji,
        'couleur'    : couleur,
        'causes'     : causes
    }

analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)

print(f"Date analysée  : {analyse['date']}")
print(f"Statut         : {analyse['statut_emoji']} {analyse['statut']}")
print(f"KPI réel       : {analyse['kpi_reel']:.3f} m³/véh")
print(f"KPI prédit     : {analyse['kpi_pred']:.3f} m³/véh")
print(f"TCM            : {analyse['tcm']:.0f} véhicules")
print()
print("Top 3 causes SHAP :")
for c in analyse['causes']:
    print(f"  {c['direction']} le KPI — {c['feature']} = {c['valeur']:.1f}  (SHAP={c['shap']:+.3f})")

Date analysée  : 2026-04-22
Statut         : 🟢 NORMAL
KPI réel       : 1.076 m³/véh
KPI prédit     : 1.046 m³/véh
TCM            : 1386 véhicules

Top 3 causes SHAP :
  ↓ réduit le KPI — Production (TCM) = 1386.0  (SHAP=-0.122)
  ↑ augmente le KPI — Eau industrielle = 1055.0  (SHAP=+0.104)
  ↑ augmente le KPI — Eau potable = 436.0  (SHAP=+0.033)


In [9]:
def predire_7_jours(model_prophet, df_prod):
    """
    Génère les prédictions à partir d'AUJOURD'HUI
    et non depuis la dernière date du dataset.
    """
    from datetime import date
    import pandas as pd

    tcm_moyen = df_prod['TCM'].mean()

    df_prophet = df_prod[['Date','KPI_m3_veh','TCM']].copy()
    df_prophet.columns = ['ds','y','TCM']
    df_prophet = df_prophet.dropna(subset=['y'])

    # ── CORRECTION : calculer combien de jours ouvrés
    # manquent entre la dernière date et aujourd'hui + 7 jours
    derniere_date = df_prophet['ds'].max()
    aujourd_hui   = pd.Timestamp(date.today())

    # Générer tous les jours ouvrés depuis la dernière date jusqu'à +7j futurs
    toutes_dates = pd.bdate_range(
        start=derniere_date + pd.Timedelta(days=1),
        end=aujourd_hui + pd.Timedelta(days=10)  # marge suffisante
    )

    # Garder uniquement les 7 prochains jours ouvrés FUTURS
    jours_futurs = pd.DataFrame({'ds': toutes_dates})
    jours_futurs = jours_futurs[jours_futurs['ds'] > aujourd_hui].head(7)

    # Construire le dataframe future complet pour Prophet
    df_future = pd.concat([
        df_prophet[['ds','TCM']],
        jours_futurs.assign(TCM=tcm_moyen)
    ], ignore_index=True)

    # Prédire
    forecast = model_prophet.predict(df_future)

    # Garder uniquement les 7 jours futurs
    predictions = forecast[forecast['ds'].isin(jours_futurs['ds'])][
        ['ds','yhat','yhat_lower','yhat_upper']
    ].copy()
    predictions.columns = ['date','kpi_predit','borne_basse','borne_haute']
    predictions['statut'] = predictions['kpi_predit'].apply(
        lambda x: '🔴 Alerte'    if x > SEUIL_ALERTE
                  else ('🟡 Attention' if x > SEUIL_OBJECTIF
                  else '🟢 Normal')
    )

    return predictions

predictions_7j = predire_7_jours(model_prophet, df_prod)

print("=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===")
print(predictions_7j.round(3).to_string(index=False))

=== PRÉDICTIONS 7 PROCHAINS JOURS RÉELS ===
      date  kpi_predit  borne_basse  borne_haute      statut
2026-04-28       1.343        1.136        1.545 🟡 Attention
2026-04-29       1.421        1.211        1.631 🟡 Attention
2026-04-30       1.396        1.180        1.608 🟡 Attention
2026-05-01       1.326        1.130        1.541 🟡 Attention
2026-05-04       1.362        1.155        1.578 🟡 Attention
2026-05-05       1.385        1.171        1.596 🟡 Attention
2026-05-06       1.462        1.249        1.676    🔴 Alerte


In [10]:
def construire_email_html(analyse, predictions_7j):

    # ── BLOC SHAP (inchangé) ─────────────────────────────────
    lignes_shap = ""
    for c in analyse['causes']:
        couleur_shap = "#A32D2D" if c['shap'] > 0 else "#27500A"
        lignes_shap += f"""
        <tr>
            <td style="padding:7px 12px;">{c['feature']}</td>
            <td style="padding:7px 12px; text-align:center;">{c['valeur']:.1f}</td>
            <td style="padding:7px 12px; text-align:center;
                       color:{couleur_shap}; font-weight:500;">
                {c['shap']:+.3f}
            </td>
            <td style="padding:7px 12px;">{c['direction']} le KPI</td>
        </tr>"""

    # ── BLOC PRÉVISIONS 7 JOURS (nouveau) ───────────────────
    lignes_prev = ""
    for _, row in predictions_7j.iterrows():

        # Couleurs selon statut
        if "Alerte" in row['statut']:
            bg_row = "#FCEBEB"
            bg_kpi = "#A32D2D"
            msg    = "Consommation prévue trop élevée — intervention recommandée"
            icone  = "🔴"
        elif "Attention" in row['statut']:
            bg_row = "#FAEEDA"
            bg_kpi = "#854F0B"
            msg    = "Proche du seuil — à surveiller"
            icone  = "🟡"
        else:
            bg_row = "#EAF3DE"
            bg_kpi = "#27500A"
            msg    = "Consommation prévue normale"
            icone  = "🟢"

        # Date en français
        jours_fr = {
            'Monday':'Lundi','Tuesday':'Mardi','Wednesday':'Mercredi',
            'Thursday':'Jeudi','Friday':'Vendredi',
            'Saturday':'Samedi','Sunday':'Dimanche'
        }
        mois_fr = {
            1:'Jan',2:'Fév',3:'Mar',4:'Avr',5:'Mai',6:'Jun',
            7:'Jul',8:'Aoû',9:'Sep',10:'Oct',11:'Nov',12:'Déc'
        }
        nom_jour = jours_fr.get(row['date'].strftime('%A'), row['date'].strftime('%A'))
        num_jour = row['date'].strftime('%d')
        nom_mois = mois_fr.get(row['date'].month, '')
        date_str = f"{nom_jour} {num_jour} {nom_mois}"

        # Barre visuelle (max visuel = 2.0 m³/véh)
        kpi_val = max(0, row['kpi_predit'])
        pct     = min(int((kpi_val / 2.0) * 100), 100)
        pct_obj = min(int((1.25   / 2.0) * 100), 100)

        lignes_prev += f"""
        <tr style="border-bottom:1px solid #e0e0e0;">
          <td style="padding:12px 14px; background:{bg_row};
                     border-left:4px solid {bg_kpi}; width:130px;">
            <div style="font-weight:600; font-size:13px; color:#222;">
              {date_str}
            </div>
          </td>
          <td style="padding:12px 14px; background:white; width:200px;">
            <div style="font-size:11px; color:#888; margin-bottom:4px;">
              Consommation prévue
            </div>
            <div style="font-size:18px; font-weight:700; color:{bg_kpi};">
              {kpi_val:.2f}
              <span style="font-size:12px; font-weight:400; color:#666;">
                m³/véhicule
              </span>
            </div>
            <div style="margin-top:6px; background:#eee;
                        border-radius:4px; height:8px; position:relative;">
              <div style="width:{pct}%; background:{bg_kpi};
                          height:8px; border-radius:4px;">
              </div>
              <div style="position:absolute; left:{pct_obj}%; top:-3px;
                          width:2px; height:14px; background:orange;">
              </div>
            </div>
            <div style="font-size:10px; color:#aaa; margin-top:2px;">
              ▲ Objectif : 1.25 m³/véh
            </div>
          </td>
          <td style="padding:12px 14px; background:white; width:160px;">
            <div style="font-size:11px; color:#888; margin-bottom:4px;">
              Fourchette probable
            </div>
            <div style="font-size:13px; color:#444;">
              entre <b>{max(0,row['borne_basse']):.2f}</b>
              et <b>{max(0,row['borne_haute']):.2f}</b> m³/véh
            </div>
            <div style="font-size:11px; color:#aaa; margin-top:2px;">
              (intervalle de confiance 80%)
            </div>
          </td>
          <td style="padding:12px 14px; background:white;">
            <div style="font-size:13px;">
              {icone} {msg}
            </div>
          </td>
        </tr>"""

    # ── HTML COMPLET ─────────────────────────────────────────
    html = f"""
    <html><body style="font-family:Arial,sans-serif;
                       max-width:700px; margin:auto; color:#222;">

      <!-- EN-TÊTE -->
      <div style="background:#1a1a2e; padding:20px 24px;
                  border-radius:8px 8px 0 0;">
        <h2 style="color:white; margin:0;">🌊 Rapport Eau Quotidien</h2>
        <p style="color:#aaa; margin:4px 0 0;">
          Renault Tanger — {analyse['date'].strftime('%A %d %B %Y')}
        </p>
      </div>

      <!-- STATUT DU JOUR -->
      <div style="background:{analyse['couleur']}; padding:16px 24px;
                  border-left:4px solid #333; margin-top:2px;">
        <h3 style="margin:0 0 10px;">
          {analyse['statut_emoji']} Statut du jour : {analyse['statut']}
        </h3>
        <table style="width:100%; border-collapse:collapse;">
          <tr>
            <td style="padding:4px 0; width:200px;">
              <b>KPI réel</b>
            </td>
            <td style="padding:4px 0;">
              <b>{analyse['kpi_reel']:.3f} m³/véhicule</b>
            </td>
            <td style="padding:4px 0; width:200px;">
              Objectif 2026
            </td>
            <td style="padding:4px 0;">
              {SEUIL_OBJECTIF} m³/véhicule
            </td>
          </tr>
          <tr>
            <td style="padding:4px 0;">KPI prédit (XGBoost)</td>
            <td style="padding:4px 0;">
              {analyse['kpi_pred']:.3f} m³/véhicule
            </td>
            <td style="padding:4px 0;">Production TCM</td>
            <td style="padding:4px 0;">
              {int(analyse['tcm'])} véhicules
            </td>
          </tr>
          <tr>
            <td style="padding:4px 0;">Consommation totale</td>
            <td style="padding:4px 0;">
              {analyse['conso']:.0f} m³
            </td>
            <td></td><td></td>
          </tr>
        </table>
      </div>

      <!-- EXPLICATION SHAP -->
      <div style="padding:16px 24px; background:#f9f9f9; margin-top:2px;">
        <h3 style="margin:0 0 12px;">
          🧠 Explication IA (SHAP) — Pourquoi ce KPI ?
        </h3>
        <table style="width:100%; border-collapse:collapse;
                      font-size:13px; background:white; border-radius:6px;">
          <thead>
            <tr style="background:#eee;">
              <th style="padding:8px 12px; text-align:left;">Variable</th>
              <th style="padding:8px 12px;">Valeur</th>
              <th style="padding:8px 12px;">Impact SHAP</th>
              <th style="padding:8px 12px; text-align:left;">Effet</th>
            </tr>
          </thead>
          <tbody>{lignes_shap}</tbody>
        </table>
      </div>

      <!-- PRÉVISIONS 7 JOURS -->
      <div style="padding:16px 24px; margin-top:2px;">
        <h3 style="margin:0 0 4px;">
          📅 Prévisions — 7 prochains jours ouvrés
        </h3>
        <p style="font-size:12px; color:#888; margin:0 0 12px;">
          Basé sur le modèle de prévision Prophet —
          Objectif Renault 2026 :
          <b>1.25 m³/véhicule</b>
          <span style="display:inline-block; width:10px; height:10px;
                       background:orange; vertical-align:middle;
                       margin-left:4px;">
          </span>
        </p>
        <table style="width:100%; border-collapse:collapse;
                      border-radius:8px; overflow:hidden;
                      box-shadow:0 1px 4px rgba(0,0,0,0.08);">
          <thead>
            <tr style="background:#1a1a2e; color:white;">
              <th style="padding:10px 14px; text-align:left;
                         font-weight:500;">Jour</th>
              <th style="padding:10px 14px; text-align:left;
                         font-weight:500;">Consommation prévue</th>
              <th style="padding:10px 14px; text-align:left;
                         font-weight:500;">Fourchette probable</th>
              <th style="padding:10px 14px; text-align:left;
                         font-weight:500;">Statut</th>
            </tr>
          </thead>
          <tbody>{lignes_prev}</tbody>
        </table>
        <div style="display:flex; gap:20px; margin-top:10px;
                    font-size:11px; color:#888;">
          <span>🟢 Normal = sous l'objectif</span>
          <span>🟡 Attention = proche du seuil</span>
          <span>🔴 Alerte = dépasse l'objectif de +15%</span>
        </div>
      </div>

      <!-- PIED DE PAGE -->
      <div style="background:#f0f0f0; padding:12px 24px;
                  border-radius:0 0 8px 8px;
                  font-size:11px; color:#888; margin-top:2px;">
        <p style="margin:0;">
          Rapport généré automatiquement — Système IA Eau
          | Renault Tanger Melloussa<br>
          PFE 2026 — Ne pas répondre à cet email
        </p>
      </div>

    </body></html>
    """
    return html

email_html = construire_email_html(analyse, predictions_7j)
print("Email HTML construit ✓")
print(f"Taille : {len(email_html)} caractères")

Email HTML construit ✓
Taille : 20008 caractères


In [11]:
def envoyer_email(html, analyse, config):
    """
    Envoie l'email via Gmail SMTP.
    """
    sujet = (
        f"[🔴 ALERTE EAU] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
        if analyse['statut'] == 'ANOMALIE'
        else f"[🟢 Rapport Eau] {analyse['date']} — KPI={analyse['kpi_reel']:.3f} m³/véh"
    )

    msg = MIMEMultipart('alternative')
    msg['Subject'] = sujet
    msg['From']    = config['expediteur']
    msg['To']      = ', '.join(config['destinataires'])
    msg.attach(MIMEText(html, 'html'))

    try:
        with smtplib.SMTP(config['smtp_serveur'], config['smtp_port']) as serveur:
            serveur.starttls()
            serveur.login(config['expediteur'], config['mot_de_passe'])
            serveur.sendmail(
                config['expediteur'],
                config['destinataires'],
                msg.as_string()
            )
        print(f"✅ Email envoyé avec succès !")
        print(f"   Sujet : {sujet}")
        print(f"   À     : {config['destinataires']}")
    except Exception as e:
        print(f"❌ Erreur envoi : {e}")
        print("   Vérifie le mot de passe application Gmail")

# Test d'envoi — décommente quand prêt
# envoyer_email(email_html, analyse, CONFIG_EMAIL)
print("Fonction prête ✓")
print("→ Décommente la dernière ligne pour envoyer l'email")

Fonction prête ✓
→ Décommente la dernière ligne pour envoyer l'email


In [12]:
# Sauvegarder l'email en HTML pour le prévisualiser dans le navigateur
with open("../outputs/email_rapport_eau.html", "w", encoding="utf-8") as f:
    f.write(email_html)

print("Email sauvegardé → ../outputs/email_rapport_eau.html")
print()
print("Pour prévisualiser :")
print("  Double-clique sur le fichier email_rapport_eau.html")
print("  Il s'ouvre dans ton navigateur exactement comme il sera reçu")

Email sauvegardé → ../outputs/email_rapport_eau.html

Pour prévisualiser :
  Double-clique sur le fichier email_rapport_eau.html
  Il s'ouvre dans ton navigateur exactement comme il sera reçu


In [13]:
def rapport_quotidien_complet():
    """
    Fonction principale appelée chaque matin par le cron.
    Regroupe tout : analyse + prédictions + email.
    """
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Démarrage rapport quotidien...")

    # 1. Analyser le jour courant
    analyse = analyser_jour_courant(df_xgb, shap_values, model_xgb)
    print(f"  Statut : {analyse['statut_emoji']} {analyse['statut']}")

    # 2. Prédictions 7 jours
    predictions = predire_7_jours(model_prophet, df_prod)
    print(f"  Prédictions 7j générées ✓")

    # 3. Construire email
    html = construire_email_html(analyse, predictions)
    print(f"  Email HTML construit ✓")

    # 4. Envoyer seulement si anomalie OU heure = 8h
    heure_actuelle = datetime.now().hour
    if analyse['statut'] == 'ANOMALIE':
        print("  ⚠️  Anomalie détectée → envoi immédiat")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    elif heure_actuelle == 8:
        print("  📧 Rapport matinal → envoi")
        envoyer_email(html, analyse, CONFIG_EMAIL)
    else:
        print("  ℹ️  Pas d'anomalie + heure != 8h → email non envoyé")
        print("      (sauvegardé dans outputs/email_rapport_eau.html)")

    return analyse, predictions

# Lancer le rapport
analyse_finale, pred_finale = rapport_quotidien_complet()

[15:50:11] Démarrage rapport quotidien...
  Statut : 🟢 NORMAL
  Prédictions 7j générées ✓
  Email HTML construit ✓
  ℹ️  Pas d'anomalie + heure != 8h → email non envoyé
      (sauvegardé dans outputs/email_rapport_eau.html)
